# 05. Product calibration and action support

Calibrates product-level behavioral draws, validates held-out event paths, and constructs product-specific empirically supported discount actions.

## 1. Imports and configuration

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        root
        for root in [CURRENT_DIR, *CURRENT_DIR.parents]
        if (root / "pyproject.toml").is_file()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(f"Could not locate project root from {CURRENT_DIR}")

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from price_of_extrapolation.action_support import (
    SupportedActionConfig,
    build_product_supported_action_sets,
)
from price_of_extrapolation.calibration import (
    CalibrationConfig,
    calibrate_products,
    load_clean_history,
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
DEMAND_ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "demand"
CALIBRATION_ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "calibration"
TABLE_DIR = PROJECT_ROOT / "results" / "final" / "tables"
FIGURE_DIR = PROJECT_ROOT / "results" / "final" / "figures"
MODEL_DIR = PROJECT_ROOT / "results" / "models"

for directory in [
    PROCESSED_DIR,
    DEMAND_ARTIFACT_DIR,
    CALIBRATION_ARTIFACT_DIR,
    TABLE_DIR,
    FIGURE_DIR,
    MODEL_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)


In [ ]:
DATA_PATH = (
    PROCESSED_DIR
    / "cereal_demand_model_data.parquet"
)
PREDICTION_PATH = (
    DEMAND_ARTIFACT_DIR
    / "demand_predictions.pkl"
)
POOLED_DRAW_PATH = (
    CALIBRATION_ARTIFACT_DIR
    / "pooled_behavioral_draws.pkl"
)

PRODUCT_DRAW_PATH = (
    CALIBRATION_ARTIFACT_DIR
    / "product_behavioral_draws.pkl"
)
PRODUCT_BASE_PATH = (
    CALIBRATION_ARTIFACT_DIR
    / "product_behavioral_bootstrap.pkl"
)
PRODUCT_CALIBRATION_ARTIFACT_PATH = (
    CALIBRATION_ARTIFACT_DIR
    / "product_calibration.pkl"
)
PRODUCT_ACTION_SET_PATH = (
    CALIBRATION_ARTIFACT_DIR
    / "supported_actions.pkl"
)

PRODUCT_SUMMARY_PATH = (
    TABLE_DIR
    / "product_calibration_summary.csv"
)
PRODUCT_EVENT_PATH = (
    TABLE_DIR
    / "product_promotion_event_study.csv"
)
PRODUCT_HOLDOUT_PATH = (
    TABLE_DIR
    / "product_holdout_event_validation.csv"
)
PRODUCT_HOLDOUT_SUMMARY_PATH = (
    TABLE_DIR
    / "product_holdout_validation_summary.csv"
)
PRODUCT_ACTION_SUPPORT_PATH = (
    TABLE_DIR
    / "supported_action_clusters.csv"
)

HOLDOUT_FIGURE_PATH = (
    FIGURE_DIR
    / "product_holdout_validation.png"
)
ACTION_FIGURE_PATH = (
    FIGURE_DIR
    / "product_supported_action_sets.png"
)

calibration_config = (
    CalibrationConfig(
        max_products=8,
        bootstrap_replications=200,
        random_seed=42,
    )
)

support_config = (
    SupportedActionConfig(
        bin_width=0.05,
        minimum_depth=0.03,
        maximum_depth=0.60,
        minimum_observations=15,
        minimum_panels=3,
        maximum_positive_actions=3,
        matching_tolerance=0.03,
    )
)

## 2. Load the selected sample and rerun product calibration

In [ ]:
if not POOLED_DRAW_PATH.is_file():
    raise FileNotFoundError(
        "Run the pooled stockpiling calibration first."
    )

loaded = load_clean_history(
    data_path=DATA_PATH,
    prediction_path=(
        PREDICTION_PATH
    ),
    config=(
        calibration_config
    ),
)

history = loaded[
    "history"
]
selected_products = loaded[
    "selected_products"
]
product_names = loaded[
    "product_names"
]
pooled_draws = pd.read_pickle(
    POOLED_DRAW_PATH
)

calibration = calibrate_products(
    history=history,
    selected_products=(
        selected_products
    ),
    product_names=(
        product_names
    ),
    pooled_draws=(
        pooled_draws
    ),
    config=(
        calibration_config
    ),
)

product_draws = calibration[
    "product_draws"
]
product_base_draws = calibration[
    "product_base_draws"
]
product_summary = calibration[
    "product_summary"
]
product_event_study = calibration[
    "product_event_study"
]
product_holdout = calibration[
    "product_holdout"
]
product_holdout_summary = calibration[
    "product_holdout_summary"
]

display(
    product_summary
)
display(
    product_holdout_summary
)

## 3. Construct product-specific supported promotion actions

In [ ]:
(
    support_table,
    supported_action_sets,
) = (
    build_product_supported_action_sets(
        history=history,
        selected_products=(
            selected_products
        ),
        product_names=(
            product_names
        ),
        config=(
            support_config
        ),
    )
)

action_set_table = pd.DataFrame(
    [
        {
            "upc": upc,
            "product_name": (
                product_names[
                    upc
                ]
            ),
            "supported_actions": (
                "|".join(
                    f"{action:.2f}"
                    for action in (
                        actions
                    )
                )
            ),
            "positive_action_count": (
                len(
                    actions
                )
                - 1
            ),
        }
        for upc, actions in (
            supported_action_sets.items()
        )
    ]
)

display(
    support_table.loc[
        support_table[
            "selected_for_grid"
        ]
    ].sort_values(
        [
            "upc",
            "action",
        ]
    )
)
display(
    action_set_table
)

## 4. Corrected holdout diagnostics

In [ ]:
if product_holdout.empty:
    raise RuntimeError(
        "No holdout promotion events were available."
    )

current_week = (
    product_holdout.loc[
        product_holdout[
            "relative_week"
        ].eq(0)
    ]
    .copy()
)

post_weeks = (
    product_holdout.loc[
        product_holdout[
            "relative_week"
        ].between(
            1,
            4,
        )
    ]
    .copy()
)

holdout_diagnostic = pd.DataFrame(
    {
        "diagnostic": [
            "current_week_MAE",
            "post_week_MAE",
            "all_week_MAE",
        ],
        "value": [
            current_week[
                "absolute_error"
            ].mean(),
            post_weeks[
                "absolute_error"
            ].mean(),
            product_holdout[
                "absolute_error"
            ].mean(),
        ],
    }
)

display(
    current_week[
        [
            "upc",
            "product_name",
            "observed_effect",
            "predicted_effect",
            "absolute_error",
            "events",
        ]
    ]
)
display(
    holdout_diagnostic
)

average_path = (
    product_holdout.groupby(
        "relative_week",
        observed=True,
    )
    .agg(
        observed_effect=(
            "observed_effect",
            "mean",
        ),
        predicted_effect=(
            "predicted_effect",
            "mean",
        ),
        predicted_q10=(
            "predicted_q10",
            "mean",
        ),
        predicted_q90=(
            "predicted_q90",
            "mean",
        ),
    )
    .reset_index()
)

fig, ax = plt.subplots(
    figsize=(
        8,
        4.8,
    )
)
ax.plot(
    average_path[
        "relative_week"
    ],
    average_path[
        "observed_effect"
    ],
    marker="o",
    label="Observed holdout",
)
ax.plot(
    average_path[
        "relative_week"
    ],
    average_path[
        "predicted_effect"
    ],
    marker="o",
    label="Predicted",
)
ax.fill_between(
    average_path[
        "relative_week"
    ],
    average_path[
        "predicted_q10"
    ],
    average_path[
        "predicted_q90"
    ],
    alpha=0.20,
)
ax.axhline(
    0.0,
    linewidth=1.0,
)
ax.axvline(
    0.0,
    linestyle="--",
    linewidth=1.0,
)
ax.set_xlabel(
    "Week relative to promotion"
)
ax.set_ylabel(
    "Residual log-demand effect"
)
ax.set_title(
    "Corrected temporal holdout validation"
)
ax.legend()
fig.tight_layout()
fig.savefig(
    HOLDOUT_FIGURE_PATH,
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 5. Supported-action figure

In [ ]:
selected_support = (
    support_table.loc[
        support_table[
            "selected_for_grid"
        ]
        & support_table[
            "action"
        ].gt(0)
    ]
    .copy()
)

fig, ax = plt.subplots(
    figsize=(
        9,
        5,
    )
)

if not selected_support.empty:
    for (
        product_name,
        group,
    ) in selected_support.groupby(
        "product_name",
        observed=True,
    ):
        ax.scatter(
            group[
                "action"
            ],
            [
                product_name
            ]
            * len(
                group
            ),
            s=(
                20
                + 3
                * np.sqrt(
                    group[
                        "support_count"
                    ]
                )
            ),
        )

ax.set_xlabel(
    "Empirically supported discount depth"
)
ax.set_ylabel(
    "Product"
)
ax.set_title(
    "Product-specific supported promotion actions"
)
fig.tight_layout()
fig.savefig(
    ACTION_FIGURE_PATH,
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 6. Save calibration outputs

In [ ]:
product_draws.to_pickle(
    PRODUCT_DRAW_PATH
)
product_base_draws.to_pickle(
    PRODUCT_BASE_PATH
)
product_summary.to_csv(
    PRODUCT_SUMMARY_PATH,
    index=False,
)
product_event_study.to_csv(
    PRODUCT_EVENT_PATH,
    index=False,
)
product_holdout.to_csv(
    PRODUCT_HOLDOUT_PATH,
    index=False,
)
product_holdout_summary.to_csv(
    PRODUCT_HOLDOUT_SUMMARY_PATH,
    index=False,
)
support_table.to_csv(
    PRODUCT_ACTION_SUPPORT_PATH,
    index=False,
)

action_artifact = {
    "selected_products": (
        selected_products
    ),
    "product_names": (
        product_names
    ),
    "supported_action_sets": (
        supported_action_sets
    ),
    "support_table": (
        support_table
    ),
    "support_config": (
        support_config
    ),
}

pd.to_pickle(
    action_artifact,
    PRODUCT_ACTION_SET_PATH,
)

calibration_artifact = {
    "selected_products": (
        selected_products
    ),
    "product_names": (
        product_names
    ),
    "calibration_end_week": (
        loaded[
            "calibration_end_week"
        ]
    ),
    "evaluation_start_week": (
        loaded[
            "evaluation_start_week"
        ]
    ),
    "holdout_diagnostic": (
        holdout_diagnostic
    ),
    "supported_action_sets": (
        supported_action_sets
    ),
}

pd.to_pickle(
    calibration_artifact,
    PRODUCT_CALIBRATION_ARTIFACT_PATH,
)

print(
    "Saved corrected product draws:",
    PRODUCT_DRAW_PATH,
)
print(
    "Saved supported actions:",
    PRODUCT_ACTION_SET_PATH,
)

## Interpretation guardrails

- The corrected current-week prediction now matches the calibration
  scale.
- Supported actions are data-dependent feasibility choices, not claims
  that unsupported depths are impossible.
- A product with no supported positive depth remains at the reference
  action in the support-constrained planner.